# TAPAS large (WTQ) — DIMER table question answering tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tapas-table-question-answering-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tapas-table-question-answering-pipeline/blob/main/tutorials/tapas_table_qa_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Ftapas--large--finetuned--wtq-ffcc4d?style=flat)](https://huggingface.co/google/tapas-large-finetuned-wtq) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Ftapas-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/tapas) [![arXiv](https://img.shields.io/badge/arXiv-2004.02349-b31b1b.svg)](https://arxiv.org/abs/2004.02349)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** table question answering (one table of string cells + one question → selected cells, one aggregation operator NONE/SUM/AVERAGE/COUNT, and a pipeline-computed numeric answer) using the pinned TAPAS-large WTQ weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/tapas_table_qa_pipeline/pipeline.py` at revision `9c9a36438890`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `f58317ab2577d17647d9acafa790c744a0388b30` (~1347 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the tokenizer flattens the table to `[CLS] question [SEP] header row + data rows [SEP]` (lower-cased WordPiece, with row, column and numeric-rank ids per token) and one forward pass of the 24-layer BERT-style encoder runs; a cell-selection head scores every token and a cell is **selected** when the mean sigmoid probability over its tokens exceeds `CELL_THRESHOLD` (0.5, the upstream default), while an aggregation head picks **one** of `NONE`, `SUM`, `AVERAGE`, `COUNT` by `argmax`. Those two decisions are exactly what the upstream `TapasTokenizer.convert_logits_to_predictions` produces and are all the model emits. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What the upstream checkpoint supplies is the encoder, the two heads and the tokenizer; what the carried pipeline module adds is manifest verification, table and question validation with named ceilings (`MAX_ROWS`, `MAX_COLUMNS`, `MAX_TOKENS`, …), the `answer` method with a fixed output contract, the `validate_inputs`/`evaluation_report` stage helpers, a `denotation_accuracy` metric, and **the numeric answer itself: `numeric_answer` is computed by the pipeline from the selected cell strings** (COUNT = number of cells; SUM/AVERAGE parse each cell back to a number, else `None`) and is labelled as such in `numeric_answer_source`. The model never emits a number.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author a synthetic table of string cells with three questions (or upload your own CSV and questions), stage and digest-verify the immutable upstream snapshot, surface the pipeline's ceilings and validate the table and questions into an input manifest, run table question answering and read the cell/aggregation/numeric contract correctly, read the machine-readable evaluation report (denotation accuracy on the author's gold answers as a sanity check, `not-measurable` without them), and export the answers as CSV plus a provenance JSON.

**This notebook does not demonstrate:** conversational or multi-turn table QA (the SQA setting), tables with more than `MAX_ROWS` rows or `MAX_COLUMNS` columns, non-English tables, text-to-SQL or arbitrary computation beyond the four aggregation operators, fine-tuning, batch questions in one forward pass, or raw-logit access. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (also float32; the pipeline loads the checkpoint in float32 on both). The model card's CPU smoke loaded and verified the 1.35 GB snapshot in 6.83 s and answered three questions on a 4×3 table in 0.95 s, 0.50 s and 0.51 s, so the default runs in well under a minute on a hosted CPU runtime once the download finishes. The pinned `torch==2.14.0` install and the 1.35 GB `model.safetensors` are the largest downloads of the run.
- **Knowledge:** basic Python; what a sigmoid threshold and an argmax are and why neither is a calibrated probability; that an aggregation over selected cells is arithmetic the pipeline performs, not a model output.
- **Data:** the default sample is one synthetic 4-row × 3-column table and three questions authored in code, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 CSV whose first row is the header and whose every cell is read as text, plus your questions typed into the form field. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded tables remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/tapas-large-finetuned-wtq` snapshot (~1347 MB in total) at revision `f58317ab2577…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `pandas` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pandas==3.0.5',
]
NOTEBOOK_SOURCE = {
    'repository': 'tapas-table-question-answering-pipeline',
    'repository_revision': '9c9a3643889026c1935767bd613eafa70e3bf26c',
    'embedded_module': 'src/tapas_table_qa_pipeline/pipeline.py',
    'embedded_modules': ['src/tapas_table_qa_pipeline/pipeline.py'],
    'module_sha256': '244b594148731261e6d1981247a2e99ed756158ceda839c206e143b1bc5246a2',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, pandas
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pandas': pandas.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/tapas_table_qa_pipeline/` @ `9c9a36438890`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/tapas_table_qa_pipeline/pipeline.py`

In [ ]:
"""Table question answering over the pinned ``google/tapas-large-finetuned-wtq`` checkpoint.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. One task method: ``answer(table, query)`` — the model
selects table cells and one aggregation operator (NONE/SUM/AVERAGE/COUNT) exactly as the upstream
``TapasTokenizer.convert_logits_to_predictions`` does; the numeric value for SUM/AVERAGE/COUNT is then
computed *by this module* from the selected cell strings and labelled as such.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "google/tapas-large-finetuned-wtq"
MODEL_REVISION = "f58317ab2577d17647d9acafa790c744a0388b30"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "tapas-large-wtq"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. MAX_ROWS/MAX_COLUMNS are ``max_num_rows``/``max_num_columns`` in the pinned config.json;
# MAX_TOKENS is ``model_max_length`` in tokenizer_config.json and the fine-tuning sequence length (README).
# A table over MAX_ROWS or MAX_COLUMNS is rejected. A table that fits those but tokenises past MAX_TOKENS
# is TRUNCATED by the tokenizer (``drop_rows_to_fit``): cell texts are first capped at a common token
# count, then trailing rows are dropped until the flattened table fits; the result reports ``rows_kept``.
MAX_ROWS = 64
MAX_COLUMNS = 32
MAX_TOKENS = 512
MAX_QUERY_CHARS = 500
MAX_CELL_CHARS = 200
# ``aggregation_labels`` in config.json, index order; the aggregation head is an argmax over these four.
AGGREGATIONS = ("NONE", "SUM", "AVERAGE", "COUNT")
# ``cell_classification_threshold`` default in the upstream convert_logits_to_predictions: a cell is
# selected when the mean sigmoid probability over its tokens exceeds this value.
CELL_THRESHOLD = 0.5
DECISION_RULE = (
    f"cell selected when its mean token sigmoid probability > CELL_THRESHOLD={CELL_THRESHOLD}; aggregation "
    "operator = argmax over the four aggregation logits; numeric answer computed by the pipeline from the "
    "selected cells (COUNT = number of cells; SUM/AVERAGE parse each cell as a number, else None)"
)
NUMERIC_ANSWER_SOURCE = "computed by the pipeline from the selected cells, not emitted by the model"
_NUMBER = re.compile(r"^[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?%?$")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _check_cell(value: Any, where: str) -> str:
    if not isinstance(value, str):
        raise TypeError(f"{where} must be str (stringify numbers yourself), got {type(value).__name__}")
    if len(value) > MAX_CELL_CHARS:
        raise ValueError(f"{where} has {len(value)} chars; ceiling is MAX_CELL_CHARS={MAX_CELL_CHARS}")
    return value


def _check_table(table: Any) -> tuple[list[str], list[list[str]]]:
    """Normalise ``{column: [cells]}`` or ``[{column: cell}, ...]`` to (columns, rows); raise on the
    first violated ceiling. Every header and cell must already be a str: TAPAS tokenises text only."""
    if isinstance(table, Mapping):
        columns = [_check_cell(name, f"column name {name!r}") for name in table]
        for name, col in table.items():
            if isinstance(col, str | bytes) or not isinstance(col, Sequence):
                raise TypeError(f"column {name!r} must be a list of str cells, got {type(col).__name__}")
        lengths = {len(col) for col in table.values()}
        if len(columns) and len(lengths) != 1:
            raise ValueError(
                f"columns have unequal lengths {sorted(lengths)}; every column needs one cell per row"
            )
        n_rows = lengths.pop() if lengths else 0
        rows = [[table[name][i] for name in columns] for i in range(n_rows)]
    elif isinstance(table, Sequence) and not isinstance(table, str | bytes):
        if not table or not all(isinstance(row, Mapping) for row in table):
            raise TypeError(
                "table must be a non-empty {column: [cells]} mapping or a list of {column: cell} rows"
            )
        columns = [_check_cell(name, f"column name {name!r}") for name in table[0]]
        rows = []
        for i, row in enumerate(table):
            if list(row) != columns:
                raise ValueError(f"row {i} has columns {list(row)}; every row must carry exactly {columns}")
            rows.append([row[name] for name in columns])
    else:
        raise TypeError("table must be a {column: [cells]} mapping or a list of {column: cell} rows")
    if not 1 <= len(columns) <= MAX_COLUMNS:
        raise ValueError(f"table has {len(columns)} columns; ceiling is 1..MAX_COLUMNS={MAX_COLUMNS}")
    if not 1 <= len(rows) <= MAX_ROWS:
        raise ValueError(f"table has {len(rows)} rows; ceiling is 1..MAX_ROWS={MAX_ROWS}")
    checked = [[_check_cell(v, f"cell[{r}][{c}]") for c, v in enumerate(row)] for r, row in enumerate(rows)]
    return columns, checked


def _check_query(query: Any) -> str:
    if not isinstance(query, str):
        raise TypeError(f"query must be str, got {type(query).__name__}")
    if not query.strip():
        raise ValueError("query is empty")
    if len(query) > MAX_QUERY_CHARS:
        raise ValueError(f"query has {len(query)} chars; ceiling is MAX_QUERY_CHARS={MAX_QUERY_CHARS}")
    return query


def parse_number(cell: str) -> float | None:
    """Parse one cell string as a number (optional sign, thousands commas, decimals, trailing %) or None."""
    text = cell.strip()
    if not _NUMBER.match(text):
        return None
    return float(text.rstrip("%").replace(",", ""))


def compute_numeric_answer(cells: Sequence[str], aggregation: str) -> tuple[float | None, list[str]]:
    """The pipeline-computed value for an aggregation: (value, cells that did not parse as numbers)."""
    if aggregation == "COUNT":
        return float(len(cells)), []
    if aggregation not in ("SUM", "AVERAGE") or not cells:
        return None, []
    parsed = [(cell, parse_number(cell)) for cell in cells]
    unparsed = [cell for cell, value in parsed if value is None]
    if unparsed:
        return None, unparsed
    values = [value for _, value in parsed if value is not None]
    return (sum(values) if aggregation == "SUM" else sum(values) / len(values)), []


def _normalise(text: str) -> str:
    return " ".join(text.lower().split())


def denotation_match(result: Mapping[str, Any], gold: str | float | int | Sequence[str]) -> bool:
    """WTQ-style denotation match: a numeric gold is compared with the pipeline's numeric answer (or the
    single selected cell parsed as a number) to 1e-6; otherwise the selected cells and the gold strings
    must be the same multiset after lower-casing and whitespace collapse."""
    cells = [str(c) for c in result.get("cells", [])]
    if isinstance(gold, bool):
        raise TypeError("gold must be a number, a string or a sequence of strings")
    if isinstance(gold, int | float):
        gold_number: float | None = float(gold)
    else:
        gold_number = parse_number(gold) if isinstance(gold, str) else None
    predicted = result.get("numeric_answer")
    if predicted is None and len(cells) == 1 and result.get("aggregation") == "NONE":
        predicted = parse_number(cells[0])
    if gold_number is not None and predicted is not None:
        return abs(float(predicted) - gold_number) <= 1e-6
    gold_cells = [gold] if isinstance(gold, str) else [] if isinstance(gold, int | float) else list(gold)
    return sorted(_normalise(c) for c in cells) == sorted(_normalise(str(g)) for g in gold_cells)


def denotation_accuracy(results: Sequence[Mapping[str, Any]], golds: Sequence[Any]) -> float:
    """Fraction of (result, gold) pairs whose denotations match; raises when the lengths differ."""
    if len(results) != len(golds) or not results:
        raise ValueError("results and golds must be non-empty and the same length")
    return sum(denotation_match(r, g) for r, g in zip(results, golds, strict=True)) / len(results)


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one table as {column: [cells]} or [{column: cell}, ...] with every header and cell a str, plus one "
        "non-empty question str; the model answers one question per call"
    ),
    "rows": [1, MAX_ROWS],
    "columns": [1, MAX_COLUMNS],
    "tokens": [1, MAX_TOKENS],
    "query_chars": [1, MAX_QUERY_CHARS],
    "cell_chars": [0, MAX_CELL_CHARS],
    "aggregations": list(AGGREGATIONS),
    "cell_threshold": CELL_THRESHOLD,
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "the table is flattened to [CLS] question [SEP] header row + data rows [SEP], lower-cased WordPiece; "
        "a flattened sequence over MAX_TOKENS is truncated by drop_rows_to_fit (cell texts capped at a "
        "common token count, then trailing rows dropped) and the result reports truncated, rows_kept and "
        "tokens_before_truncation; cells stay "
        "strings for the tokenizer and are parsed back to numbers only for the pipeline-computed "
        "SUM/AVERAGE answer"
    ),
}


def validate_inputs(
    table: Mapping[str, Sequence[str]] | Sequence[Mapping[str, str]],
    queries: Sequence[str],
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, table and per-query observations, verdict).

    Rejection raises exactly as ``answer`` would: both route through ``_check_table`` and ``_check_query``.
    ``answer`` takes one query per call, so ``queries`` is the batch the notebook loops over against the
    same table. The token ceiling (``MAX_TOKENS``) needs the loaded tokenizer, so it is not observable here;
    ``answer`` reports ``rows_kept`` and ``truncated`` after tokenisation.
    """
    columns, rows = _check_table(table)
    if isinstance(queries, str | bytes) or not isinstance(queries, Sequence):
        raise TypeError("queries must be a sequence of str, not a single string")
    if not queries:
        raise ValueError("queries must hold at least one item")
    checked = [_check_query(q) for q in queries]
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per query")
    return {
        "schema": dict(INPUT_SCHEMA),
        "table": {
            "columns": columns,
            "n_rows": len(rows),
            "n_columns": len(columns),
            "numeric_cells": sum(parse_number(cell) is not None for row in rows for cell in row),
        },
        "inputs": [
            {"id": names[i] if names else f"query-{i}", "chars": len(q), "query": q}
            for i, q in enumerate(checked)
        ],
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Mapping[str, Any] | Sequence[Mapping[str, Any]],
    golds: Sequence[Any] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``golds`` (one gold denotation per result: a number, a string or a list of cell strings) the report
    carries ``denotation_accuracy`` as sample-sanity evidence; without them the verdict is ``not-measurable``
    and the report says what labelled data would make the task measurable.
    """
    items = [results] if isinstance(results, Mapping) else list(results)
    base = {
        "task": "table question answering (cell selection + aggregation, WikiTableQuestions fine-tune)",
        "decision_rule": DECISION_RULE,
        "sample_kind": sample_kind,
        "n_results": len(items),
        "aggregations": [r.get("aggregation") for r in items],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if golds is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no gold denotation was supplied for the evaluated questions",
            "needs": (
                "one gold denotation per question (the cell strings, or the number an aggregation should "
                "produce) over enough questions from the deployment's own tables to state a dispersion, "
                "scored with denotation_accuracy; the upstream WTQ dev figure is not reproduced here"
            ),
        }
    return {
        **base,
        "metrics": [
            {
                "id": "denotation_accuracy",
                "value": denotation_accuracy(items, list(golds)),
                "n": len(items),
                "estimation": "single sample, no dispersion estimate",
            }
        ],
        "verdict": "sample-sanity",
        "reason": (
            f"{len(items)} question(s) with author-supplied gold denotations on one sample table; not a "
            "benchmark"
        ),
        "needs": (
            "a labelled table-question set from the deployment domain for any generalisable accuracy claim"
        ),
    }


@dataclass
class TAPASTableQAPipeline:
    """``_runner(columns, rows, query)`` -> ``{coordinates: [(row, col)], aggregation_index,
    aggregation_logits, n_tokens, full_tokens, rows_kept}`` where coordinates index data rows (0 = first row
    below the header) and full_tokens is the untrimmed length; injectable so tests run offline."""

    _runner: Callable[[list[str], list[list[str]], str], Mapping[str, Any]]
    device: str = "cpu"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> TAPASTableQAPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), dict(local_files_only=True), "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, dict(revision=MODEL_REVISION), "hf-hub"
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import TapasForQuestionAnswering, TapasTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        import pandas as pd  # TapasTokenizer takes a DataFrame; imported after the refusal like the others
        tokenizer = TapasTokenizer.from_pretrained(location, trust_remote_code=False, **kwargs)
        model = TapasForQuestionAnswering.from_pretrained(
            location, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        class _PositionalRows(pd.DataFrame):
            """transformers 4.57.6's TapasTokenizer reads each ``iterrows()`` row by integer position
            (``row[col_index]``), which pandas 3 no longer accepts on a str-labelled Series; rows are yielded
            re-indexed by position so the tokenizer's numeric-value annotation runs unchanged."""

            @property
            def _constructor(self):
                return _PositionalRows

            def iterrows(self):
                for index, row in super().iterrows():
                    yield index, row.reset_index(drop=True)

        def runner(columns: list[str], rows: list[list[str]], query: str) -> dict[str, Any]:
            # dtype=object: pandas 3's default pyarrow str dtype cannot hold the tokenizer's Cell objects
            frame = _PositionalRows(rows, columns=columns, dtype=object)
            encoded = tokenizer(
                table=frame, queries=[query], truncation="drop_rows_to_fit", max_length=MAX_TOKENS,
                padding="max_length", return_tensors="pt",
            )
            with torch.inference_mode():
                out = model(**{k: v.to(resolved_device) for k, v in encoded.items()})
            coordinates, agg = tokenizer.convert_logits_to_predictions(
                encoded,
                out.logits.cpu(),
                out.logits_aggregation.cpu(),
                cell_classification_threshold=CELL_THRESHOLD,
            )
            # Untrimmed length ([CLS] query [SEP] header + cells): when it exceeds n_tokens the tokenizer
            # trimmed cell text and/or dropped rows to fit MAX_TOKENS.
            flat = [query, *columns, *(cell for row in rows for cell in row)]
            return {
                "coordinates": [(int(r), int(c)) for r, c in coordinates[0]] if coordinates else [],
                "aggregation_index": int(agg[0]),
                "aggregation_logits": out.logits_aggregation[0].float().cpu().tolist(),
                "n_tokens": int(encoded["attention_mask"].sum()),
                "full_tokens": 2 + sum(len(tokenizer.tokenize(text)) for text in flat),
                "rows_kept": int(encoded["token_type_ids"][0, :, 2].max()),
            }

        return cls(runner, resolved_device, source)

    def answer(
        self, table: Mapping[str, Sequence[str]] | Sequence[Mapping[str, str]], query: str
    ) -> dict[str, Any]:
        """Select cells and an aggregation operator for one question; compute the numeric answer from them."""
        columns, rows = _check_table(table)
        query = _check_query(query)
        raw = self._runner(columns, rows, query)
        coordinates = [(int(r), int(c)) for r, c in raw["coordinates"]]
        if any(not (0 <= r < len(rows) and 0 <= c < len(columns)) for r, c in coordinates):
            raise RuntimeError(f"backend returned coordinates outside the {len(rows)}x{len(columns)} table")
        aggregation = AGGREGATIONS[int(raw["aggregation_index"])]
        cells = [rows[r][c] for r, c in coordinates]
        value, unparsed = compute_numeric_answer(cells, aggregation)
        rows_kept = int(raw.get("rows_kept", len(rows)))
        n_tokens = int(raw.get("n_tokens", 0))
        full_tokens = int(raw.get("full_tokens", n_tokens))
        return {
            "cells": cells,
            "coordinates": [list(c) for c in coordinates],
            "aggregation": aggregation,
            "answer": (f"{aggregation} > " if aggregation != "NONE" else "") + ", ".join(cells),
            "numeric_answer": value,
            "numeric_answer_source": NUMERIC_ANSWER_SOURCE,
            "unparsed_cells": unparsed,
            "aggregation_logits": dict(zip(AGGREGATIONS, map(float, raw["aggregation_logits"]), strict=True)),
            "n_tokens": n_tokens,
            "tokens_before_truncation": full_tokens,
            "rows_kept": rows_kept,
            "truncated": rows_kept < len(rows) or n_tokens < full_tokens,
            "decision_rule": DECISION_RULE,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `6`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `f58317ab2577…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TAPASTableQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "tapas-large-wtq",
  "modelId": "google/tapas-large-finetuned-wtq",
  "revision": "f58317ab2577d17647d9acafa790c744a0388b30",
  "files": [
    {
      "path": "README.md",
      "bytes": 7224,
      "sha256": "2397bdae46316684903465e6f425a9fe10c85d491a9755ec7a31a74353c034c8"
    },
    {
      "path": "config.json",
      "bytes": 1659,
      "sha256": "834f44a325a349d2d209ecb18e4fa2ca3cdb16cef5b169e5c0a8f5e16c444d13"
    },
    {
      "path": "model.safetensors",
      "bytes": 1346985282,
      "sha256": "149247e13732c222ba621e0c4e7b90ba260869b36adcbe115dc872c2c98bccf0"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 154,
      "sha256": "e3ec7abc6bcd45aba696cb95e9945186dad920aea78733f8b45c83d696ed0dea"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 490,
      "sha256": "ab033df4f3c902cbc66bae2736bc99f4a59c43e7d9e53e40046576826b2cc5e9"
    },
    {
      "path": "vocab.txt",
      "bytes": 262028,
      "sha256": "4d96f9308bcf9019684fcc109aa8c042b9b745edabba0162fbe66c75ebee2db4"
    }
  ],
  "totalBytes": 1347256837
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TAPASTableQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the synthetic sample or optional BYOD

The default sample is **synthetic**, written in this cell: a 4-row × 3-column table of Philippine cities (the model card's smoke table) whose population cells carry thousands separators, and three questions each given a stable identifier — a cell **lookup**, a **COUNT** and a **SUM** — together with the answer the author expects. Every header and cell is a **string**: TAPAS tokenises text, so numbers must arrive as text and the pipeline parses them back only when it computes SUM/AVERAGE. The expected answers are the author's intent, not a labelled dataset: whether the model reproduces them is a sanity check that the code path works, never benchmark evidence.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 CSV whose first row is the header, at most `MAX_ROWS` data rows and `MAX_COLUMNS` columns, every cell read as text and at most `MAX_CELL_CHARS` characters; questions go in `BYOD_QUESTIONS`, separated by ` | `, each at most `MAX_QUERY_CHARS` characters. A flattened table longer than `MAX_TOKENS` WordPiece tokens is **truncated** by the tokenizer (cell text trimmed to a common token count, then trailing rows dropped) and the result reports it. The upload stays inside this runtime. If you also hold gold answers, keep them outside the notebook — Section 7 explains what to compute with them.

In [ ]:
import csv
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
BYOD_QUESTIONS = 'How many rows are there? | What is the total of the second column?'  # @param {type:"string"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    reader = list(csv.reader(io.StringIO(uploaded[sample_name].decode('utf-8'))))
    if len(reader) < 2:
        raise ValueError(f'{sample_name}: expected a header row followed by at least one data row')
    header, data_rows = reader[0], reader[1:]
    table = {name: [row[i] if i < len(row) else '' for row in data_rows] for i, name in enumerate(header)}
    queries = [q.strip() for q in BYOD_QUESTIONS.split('|') if q.strip()]
    golds = None
    sample_kind = 'BYOD upload'
else:
    table = {
        'City': ['Manila', 'Cebu', 'Davao', 'Baguio'],
        'Population (2020)': ['1,846,513', '964,169', '1,776,949', '366,358'],
        'Region': ['NCR', 'Region VII', 'Region XI', 'CAR'],
    }
    queries = [
        'Which city is in Region VII?',
        'How many cities are listed?',
        'What is the total population of Manila and Davao?',
    ]
    golds = ['Cebu', 4, 3623462]
    sample_name = 'synthetic_cities_table'
    sample_kind = 'synthetic (authored in this cell; the model card smoke table)'
query_ids = [f'q{index + 1}' for index in range(len(queries))]
sample_sha256 = hashlib.sha256(json.dumps({'table': table, 'queries': queries}, sort_keys=True).encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'columns': list(table), 'rows': len(next(iter(table.values()))), 'questions': len(queries), 'golds': golds, 'sample_sha256': sample_sha256})
for query_id, query in zip(query_ids, queries, strict=True):
    print(f'{query_id}: {query[:120]}')

## 5. Validate the inputs → input manifest

`validate_inputs` is the pipeline's public validation stage: it takes the table and the list of questions `answer` will be called with, and its checks are the method's own — `_check_table` and `_check_query` — so a rejection here is a rejection there. `MAX_ROWS` (64) and `MAX_COLUMNS` (32) are the pinned config's `max_num_rows`/`max_num_columns` and **reject** larger tables; `MAX_CELL_CHARS` and `MAX_QUERY_CHARS` are character guards applied before tokenisation; `MAX_TOKENS` (512, the checkpoint's fine-tuning sequence length) is applied by the tokenizer after tokenisation and **truncates** rather than rejects, so it is reported by `answer` (`truncated`, `rows_kept`, `tokens_before_truncation`) and cannot be observed at this stage; `AGGREGATIONS` names the four operators and `CELL_THRESHOLD` the selection cut-off. The manifest records the table shape and how many cells parse as numbers, and is written to `outputs/tapas_table_qa_input_manifest.json`. To show what rejection looks like, the cell also validates a table carrying a numeric (non-string) cell and records the pipeline's own error message as a finding. The notebook never trims or alters the table.

In [ ]:
os.makedirs('outputs', exist_ok=True)
ceilings = {'MAX_ROWS': MAX_ROWS, 'MAX_COLUMNS': MAX_COLUMNS, 'MAX_TOKENS': MAX_TOKENS, 'MAX_QUERY_CHARS': MAX_QUERY_CHARS, 'MAX_CELL_CHARS': MAX_CELL_CHARS, 'AGGREGATIONS': AGGREGATIONS, 'CELL_THRESHOLD': CELL_THRESHOLD}
print(ceilings)
input_manifest = validate_inputs(table, queries, names=query_ids)
# Demonstrate the strings-only rejection; the finding is recorded, not swallowed.
try:
    validate_inputs({'Item': ['a', 'b'], 'Units': [1, 2]}, queries)
except TypeError as exc:
    input_manifest['findings'].append({'input': 'numeric-cell-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/tapas_table_qa_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))
print({'token_ceiling': f'MAX_TOKENS={MAX_TOKENS} is applied by the tokenizer after tokenisation and truncates (cell text trimmed, then trailing rows dropped); answer() reports truncated, rows_kept and tokens_before_truncation'})

## 6. Answer the questions and read the contract correctly

**Input/output contract.** `answer(table, query)` takes the table and one question and returns `cells` (the selected cell strings, in row-major order), `coordinates` (`[row, column]` into the data rows, 0 = first row below the header), `aggregation` (one of `AGGREGATIONS`), `answer` (the upstream pipeline's string form: the cells joined with `, `, prefixed by `SUM > ` etc. when an operator was chosen), `numeric_answer` with `numeric_answer_source`, `unparsed_cells` (cells that could not be read as numbers, in which case `numeric_answer` is `None`), `aggregation_logits` (the four raw head scores), `n_tokens`, `tokens_before_truncation`, `rows_kept`, `truncated`, the decision rule, the device and the model identity. **Decision semantics:** the cell decision is a fixed sigmoid threshold (`CELL_THRESHOLD` = 0.5) and the operator decision is an `argmax` over four logits; neither is a calibrated probability, the pipeline applies no acceptance threshold on the aggregation logits, and a question the table cannot answer still yields *some* cells and *some* operator — including `SUM` over a single cell, which the model card's probe observed. **The numeric answer is the pipeline's arithmetic**, not a model output: it is exactly right when the selected cells and operator are right, and confidently wrong otherwise. The model card's CPU smoke on this table returned `Cebu` (NONE), `COUNT` over the four city cells → 4.0, and `SUM` over `1,846,513` and `1,776,949` → 3,623,462.0; those are single observations, not expected values. Inference is deterministic on a fixed device and dtype (`model.eval()`, no sampling, no seed needed).

In [ ]:
import time

results = []
n_rows = len(next(iter(table.values())))
for query_id, query in zip(query_ids, queries, strict=True):
    started = time.perf_counter()
    result = pipe.answer(table, query)
    elapsed = time.perf_counter() - started
    checks = {
        'aggregation_known': result['aggregation'] in AGGREGATIONS,
        'coordinates_inside_table': all(0 <= r < n_rows and 0 <= c < len(table) for r, c in result['coordinates']),
        'cells_match_coordinates': result['cells'] == [list(table.values())[c][r] for r, c in result['coordinates']],
        'count_equals_cells': result['aggregation'] != 'COUNT' or result['numeric_answer'] == len(result['cells']),
        'numeric_none_iff_unparsed_or_no_aggregation': (result['numeric_answer'] is None) == (bool(result['unparsed_cells']) or result['aggregation'] == 'NONE' or not result['cells']),
        'rows_kept_within_table': 0 <= result['rows_kept'] <= n_rows,
        'n_tokens_within_ceiling': 1 <= result['n_tokens'] <= MAX_TOKENS,
    }
    if not all(checks.values()):
        raise RuntimeError(f'answer output failed a sanity check for {query_id}: {checks}')
    results.append({'id': query_id, 'query': query, 'seconds': round(elapsed, 3), 'checks': checks, **result})
    print(f"{query_id}: {query}")
    print(f"    {result['aggregation']:<8} cells={result['cells']} coords={result['coordinates']} numeric_answer={result['numeric_answer']} ({result['numeric_answer_source']})")
    print(f"    tokens={result['n_tokens']}/{result['tokens_before_truncation']} rows_kept={result['rows_kept']}/{n_rows} truncated={result['truncated']} seconds={elapsed:.3f}")
print({'decision_rule': DECISION_RULE, 'numeric_answer_source': NUMERIC_ANSWER_SOURCE})
if any(r['truncated'] for r in results):
    print('At least one question ran on a truncated table: cell text was trimmed and/or trailing rows dropped to fit MAX_TOKENS; answers may refer to a table the model only partly saw.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. The repository ships one metric helper, **`denotation_accuracy`** — the WikiTableQuestions criterion: a prediction is correct when its denotation equals the gold (a numeric gold against `numeric_answer`, or the single selected cell parsed as a number; otherwise the selected cell strings against the gold strings after lower-casing and whitespace collapse). On the synthetic sample the author's three expected answers are passed as `golds`, so the verdict is **`sample-sanity`** with a single-sample estimate and no dispersion — a falsifiable plumbing check on three questions, **not** an accuracy claim and not comparable to the upstream WTQ dev figure (0.5097, upstream-reported, not measured here). On a BYOD upload no golds are supplied, so the verdict is `not-measurable` and the report states what would make it measurable: one gold denotation per question over enough questions from your own tables to state a dispersion. The report is written to `outputs/tapas_table_qa_evaluation_report.json`.

In [ ]:
report = evaluation_report(results, golds, sample_kind=sample_kind)
with open('outputs/tapas_table_qa_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'sample-sanity':
    print('denotation_accuracy on the author-supplied golds is a plumbing check on a handful of questions, not an accuracy figure; see needs.')
if report['verdict'] == 'not-measurable':
    print('No metric is reported: supply one gold denotation per question to compute denotation_accuracy on your own table-question set.')

## 8. Export answers and provenance

Two further files are written under `outputs/` beside the input manifest and the evaluation report. The answers go to CSV (`outputs/tapas_table_qa_answers.csv`) with one row per question — `id`, `query`, `aggregation`, `cells` (joined with ` | `), `coordinates`, `numeric_answer`, `numeric_answer_source`, `unparsed_cells`, `n_tokens`, `tokens_before_truncation`, `rows_kept`, `truncated`, `seconds` — so every answer stays attached to its question. One JSON record (`outputs/tapas_table_qa_result.json`) preserves the table, every result with its aggregation logits and sanity checks, the decision rule, the ceilings in force, the input manifest, the evaluation report, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, `pandas`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
with open('outputs/tapas_table_qa_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['id', 'query', 'aggregation', 'cells', 'coordinates', 'numeric_answer', 'numeric_answer_source', 'unparsed_cells', 'n_tokens', 'tokens_before_truncation', 'rows_kept', 'truncated', 'seconds'])
    for r in results:
        writer.writerow([r['id'], r['query'], r['aggregation'], ' | '.join(r['cells']), json.dumps(r['coordinates']), r['numeric_answer'], r['numeric_answer_source'], ' | '.join(r['unparsed_cells']), r['n_tokens'], r['tokens_before_truncation'], r['rows_kept'], r['truncated'], r['seconds']])
payload = {
    'table': table,
    'results': [{key: value for key, value in r.items() if key not in ('device', 'source', 'model_id', 'model_revision')} for r in results],
    'decision_rule': DECISION_RULE,
    'numeric_answer_source': NUMERIC_ANSWER_SOURCE,
    'answers_file': 'outputs/tapas_table_qa_answers.csv',
    'ceilings': ceilings,
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'name': sample_name, 'kind': sample_kind, 'sample_sha256': sample_sha256, 'golds': golds},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': snapshot['path'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'pandas': pandas.__version__,
        'device': pipe.device,
        'dtype': 'float32',
    },
}
with open('outputs/tapas_table_qa_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

TAPAS answers a question by selecting cells (a fixed 0.5 sigmoid threshold per cell) and choosing one of four aggregation operators (an argmax); it emits no number and no calibrated confidence. The `numeric_answer` you see is the pipeline's arithmetic over the selected cell strings — exact when the selection and operator are right, silently wrong when they are not, and `None` when a selected cell does not parse as a number. The model always produces some answer, including operators applied to a single cell, and applies no acceptance threshold on the aggregation logits; any cut-off is the caller's to set on labelled questions from their own tables. Tables are lower-cased WordPiece text: numbers must arrive as strings, and a flattened table over 512 tokens is truncated (cell text trimmed, then trailing rows dropped) rather than rejected, so `truncated`, `rows_kept` and `tokens_before_truncation` must be read before trusting an answer. The checkpoint was fine-tuned on English Wikipedia tables (WTQ, after SQA and WikiSQL); behaviour on other domains, languages or table shapes is not measured here. On the synthetic sample the sample-sanity denotation accuracy is a plumbing check on three questions, not an accuracy figure; a real evaluation needs gold denotations over enough questions to state a dispersion.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot, validate the demonstrated inputs against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, denotation accuracy on any domain, a usable threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments.** Ask a question the table cannot answer and read which cells and operator the model still returns; ask for an average and watch the pipeline parse the thousands separators; paste a table with a long text column and read `truncated`/`rows_kept` to see what the tokenizer dropped; assemble a dozen questions with gold denotations over one of your own tables and compute the `denotation_accuracy` the evaluation report asks for. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/tapas-table-question-answering-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/tapas-table-question-answering-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/tapas-table-question-answering-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google/tapas-large-finetuned-wtq
- Upstream code: https://github.com/google-research/tapas
- TAPAS: Weakly Supervised Table Parsing via Pre-training: https://arxiv.org/abs/2004.02349
- Understanding tables with intermediate pre-training: https://arxiv.org/abs/2010.00571
- Compositional Semantic Parsing on Semi-Structured Tables (WikiTableQuestions): https://arxiv.org/abs/1508.00305